# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane:** Freestyle — Growth / Recovery / Momentum Prediction.

**Why:** A page's trajectory comes from several signals moving at once — visibility, position, freshness, engagement — and which combination matters varies by page. A hand rule like Week 2's `stale x visible` can spot "this looks stale"; it can't weigh shifting signals against noisy history to say "momentum is turning." The useful question for FlyRank isn't "is this page bad now" but "is it heading somewhere worth acting on."

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Decision.** Which pages senior SEO specialists review first, ranked by how much each page's search
traffic is likely to change over the next 30 days.

**Actor.** Senior SEO specialists. The model ranks and explains; a specialist accepts or declines
before anything happens to the page.

**Cost, asymmetric.**
- *False flag* — a specialist spends review time on a page that didn't need it. Recoverable.
- *Missed signal* — a real decline never surfaces and the window to act closes. Not recoverable.

The missed signal costs more, so recall matters alongside precision. Both are reported **pooled**
across clients, so every page counts once regardless of which client it belongs to.

**Two outputs, not one.** A ranked queue for pages whose traffic is moving, and a **separate flag**
for pages that lose all their traffic. Total loss is an emergency rather than a ranking problem, and
a page at zero cannot be expressed as a ratio against its own past at all.

**Stagnant pages need no special handling.** A page that does not move scores near zero and lands in
the middle of the ranking on its own — neither urgent nor interesting. That falls out of the target
rather than being designed in.

---

> **Previously:** the decision was framed as three yes/no questions — will this page decline, recover,
> or gain momentum — with stagnant pages as a *negative class*. That design is superseded; the
> trichotomy became one continuous magnitude, and with no classes there is no negative class. See the
> revision log at the end.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [1]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Rows:", len(df))
print()
print("trend_direction breakdown:")
print(df["trend_direction"].value_counts())
print()
print((df["trend_direction"].value_counts(normalize=True) * 100).round(1).astype(str) + "%")
print()

# Rough gut-check only, not a settled window: this starter CSV has no daily granularity
# so it can't actually test the real prior/future window design.
# The real feasibility check happens in ML-03 against fact_content_daily_performance, once the actual window lengths are picked.
print("Pages old enough to plausibly support SOME prior+future window (content_age_days >= 180):",
      (df["content_age_days"] >= 180).sum(), f'({(df["content_age_days"] >= 180).mean():.1%})')
print("Pages visible enough to matter for review (impressions_90d >= 500):",
      (df["impressions_90d"] >= 500).sum(), f'({(df["impressions_90d"] >= 500).mean():.1%})')

Rows: 30000

trend_direction breakdown:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

trend_direction
down      54.2%
stable    19.9%
up        14.6%
new        7.5%
flat       3.8%
Name: proportion, dtype: str

Pages old enough to plausibly support SOME prior+future window (content_age_days >= 180): 17986 (60.0%)
Pages visible enough to matter for review (impressions_90d >= 500): 16726 (55.8%)


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**Can say:** observed patterns in past search and engagement data; a directional, magnitude-ranked
estimate of how much a page's traffic is likely to move; a decision-support ordering for limited
review time.

**Will never say:** that a flagged page is guaranteed to decline or recover; that editing a page
caused a recovery (that needs an experiment); anything about Google's ranking algorithm; anything
client-identifying — pseudonymous IDs and aggregates only.

**One limit worth stating early.** Review capacity binds harder than model quality. At a 100-page
monthly audit against a median 422 declines per client, even a perfect ranking reaches about **4.2%**
of them. The model decides *which* pages get looked at, not *how many* — so its value is in the
ordering, never in coverage.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.



## Revision log

The sections above state the current framing. This is what changed after Week 1 and why.

| date | change | why |
|---|---|---|
| 08-05 | three yes/no questions → **one continuous magnitude** | the ±20% cut separating them was inherited, not derived; it made a 21% dip and a 95% collapse the same label; and `future_decline` was false *by construction* for already-declining pages, which manufactured 89% of ML-06 Test 3's gradient |
| 08-05 | stagnant pages: *negative class* → **no special handling** | with no classes there is no negative class. A stagnant page scores near zero and sorts to the middle on its own |
| 08-06 | added the **separate dead-page flag** | a ratio target cannot represent a page reaching zero, and 7.4% of the D1 cohort does. Total traffic loss is an emergency, not a queue position |
| 08-06 | recall and precision reported **pooled** | averaging per-client rates once overstated recall more than tenfold |

**What did not change.** The decision, the actor, the cost asymmetry, and the careful-words section
are all as written in Week 1. The framing never depended on how the target was computed — only on
the fact that pages get ranked and a specialist decides. That is why this notebook needed the least
revision of any in the project.

Evidence for every row is in **ML-06**; the full design history is in the capstone report, §9.